# Phase 00 — Scope, environment and source registry

**Project:** Open Materials Data Pipeline for Hydrogel Bioink Design
**Source:** [Zenodo record 19602891](https://zenodo.org/records/19602891) · DOI [10.5281/zenodo.19602891](https://doi.org/10.5281/zenodo.19602891) — phenol-modified alginate (ALG-Ph) and hyaluronic acid (HA-Ph) bioinks

**What this notebook does.** It confirms that the notebook runs inside the project folder, that the Python packages match `environment.yml`, and that the source files are present and byte-identical to the files used in this project. It then records the source registry and the check results. It does not open, parse or change any data.

| | |
|---|---|
| **Inputs** | `environment.yml`; `data/raw/zenodo_19602891/ALG-Ph_HA-Ph_rheology_data.zip`; `data/raw/zenodo_19602891/image_index.csv`; `images.zip` (optional, Phase 06 only) |
| **Outputs** | `data/metadata/phase00/source_registry.csv`; `data/metadata/phase00/phase00_checks.csv`; `data/metadata/phase00/phase00_run_record.json` |
| **Runtime** | Under a minute (a few seconds longer when `images.zip` is present, for its checksum) |

**How to run**

1. Set up the environment once: `conda env create -f environment.yml`, then `conda activate hydrogel-bioink` (or `pip install -r requirements.txt`).
2. Download the source files from the repository root: `python scripts/download_data.py` (add `--images` for Phase 06).
3. Open this notebook from anywhere inside the repository, select the **Python (hydrogel-bioink)** kernel, then **Kernel → Restart Kernel and Run All Cells**.
4. Read the summary at the end. Every check reports **PASS**, **WARN** or **FAIL**. A FAIL stops the notebook with a message naming the file and the problem.

**Status labels used in this project**

| Label | Meaning |
|---|---|
| **Verified** | Confirmed by an automated check in this run |
| **Preliminary** | Observed, but not yet confirmed by a check or by source documentation |
| **Assumption** | Taken as given for the workflow; stated so it can be revisited |
| **Unresolved** | An open question; no decision has been made |

## 1. Settings

This cell holds every fixed value the notebook uses: the source record, the expected files and their reference MD5 checksums, and the output folder. The reference checksums identify the exact files audited in Phase 01, so a later change to the Zenodo record is detected here rather than in a downstream result.

In [ ]:
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path
import hashlib
import json
import platform
import re
import sys

import pandas as pd

RECORD_ID = "19602891"
RECORD_URL = f"https://zenodo.org/records/{RECORD_ID}"
RECORD_DOI = f"10.5281/zenodo.{RECORD_ID}"
RAW_DIR = Path("data") / "raw" / f"zenodo_{RECORD_ID}"   # relative to the project root
OUT_DIR = Path("data") / "metadata" / "phase00"

# file name -> (role, required for Phases 00-05?, reference MD5 or None)
SOURCE_FILES = {
    "ALG-Ph_HA-Ph_rheology_data.zip": ("92 rheology workbooks (viscosity and modulus exports)", True, "4eaac901afc46212108bfdad8b39b5c2"),
    "image_index.csv": ("Index of 1,031 printed-construct images with ink, shape and intensity metadata", True, "e0fc1a61b6f5d5b2ea87a69a97d4aa58"),
    "images.zip": ("1,031 printed-construct PNG images (Phase 06 only)", False, None),
}

checks = []  # one row per check: id, description, status, detail

def record(check_id, description, status, detail):
    # Add one check result and print it on a single line.
    checks.append({"check_id": check_id, "description": description, "status": status, "detail": detail})
    print(f"[{status:4}] {check_id}: {detail}")

print("Settings loaded.")

## 2. Locate the project folder

The notebook searches upward from its working folder for the repository markers `README.md`, `environment.yml` and `notebooks/`. All later paths are built from this folder, so the notebook gives the same result whether it is opened from the repository root or from `notebooks/`. If the markers are not found, the notebook stops: continuing from an unknown folder could read the wrong files.

Paths are printed relative to the project folder, so no personal folder names appear in the saved notebook or its HTML export.

In [ ]:
def find_project_root(start: Path) -> Path | None:
    for folder in (start, *start.parents):
        if (folder / "README.md").is_file() and (folder / "environment.yml").is_file() and (folder / "notebooks").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project folder not found: no parent folder contains README.md, environment.yml and notebooks/. "
        "Open this notebook from inside the cloned hydrogel-bioink-data-curation repository."
    )

record("P00-01", "Project folder located", "PASS", f"repository root found ('{PROJECT_ROOT.name}')")

## 3. Check the Python environment

Package versions can change parsing, numerical results and figures. This cell compares the running Python and package versions with the pinned versions in `environment.yml`. A different version is reported as **WARN**, not FAIL: the workflow may still run, but numerical outputs are only expected to match the published reports exactly under the pinned versions. A missing package is a **FAIL**, because later phases cannot run without it.

In [ ]:
env_text = (PROJECT_ROOT / "environment.yml").read_text(encoding="utf-8")
pinned = dict(re.findall(r"-\s*([A-Za-z0-9_.-]+)==([0-9][^\s#]*)", env_text))
pinned_python = re.search(r"-\s*python=([0-9.]+)", env_text).group(1)

running_python = ".".join(platform.python_version_tuple()[:2])
record("P00-02", "Python version", "PASS" if running_python == pinned_python else "WARN",
       f"running {platform.python_version()}, environment.yml pins {pinned_python}")

rows = []
for package, wanted in pinned.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    rows.append({"package": package, "pinned": wanted, "installed": installed or "not installed",
                 "status": "FAIL" if installed is None else ("PASS" if installed == wanted else "WARN")})
packages = pd.DataFrame(rows)

worst = "FAIL" if (packages.status == "FAIL").any() else "WARN" if (packages.status == "WARN").any() else "PASS"
differing = packages.loc[packages.status != "PASS", "package"].tolist()
record("P00-03", "Package versions match environment.yml", worst,
       f"{(packages.status == 'PASS').sum()} of {len(packages)} match" + (f"; differ or missing: {', '.join(differing)}" if differing else ""))
packages

## 4. Check the source files

Each source file is checked in the raw-data folder `data/raw/zenodo_19602891/`:

- **Present** — the file exists. A missing required file is a FAIL; a missing `images.zip` is a WARN, because only Phase 06 uses it.
- **Identical** — the file's MD5 checksum equals the checksum of the file audited in Phase 01. A mismatch is a FAIL: it means the file is incomplete or comes from a different release of the Zenodo record, and downstream results would not match the published reports.

No file is opened beyond reading its bytes for the checksum.

In [ ]:
def md5_of(path: Path) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

registry_rows = []
for i, (name, (role, required, reference_md5)) in enumerate(SOURCE_FILES.items(), start=4):
    path = PROJECT_ROOT / RAW_DIR / name
    present = path.is_file()
    actual_md5 = md5_of(path) if present else None
    check_id = f"P00-{i:02d}"

    if not present:
        status = "FAIL" if required else "WARN"
        detail = f"{name} not found in {RAW_DIR.as_posix()}/ — run: python scripts/download_data.py" + ("" if required else " --images")
    elif reference_md5 and actual_md5 != reference_md5:
        status, detail = "FAIL", f"{name} checksum {actual_md5} differs from the audited file ({reference_md5})"
    elif reference_md5:
        status, detail = "PASS", f"{name} present and identical to the audited file"
    else:
        status, detail = "PASS", f"{name} present ({path.stat().st_size / 1024**2:,.1f} MiB); checksum recorded"

    record(check_id, f"Source file: {name}", status, detail)
    registry_rows.append({
        "file": name, "role": role, "required_for_phases_00_05": required,
        "relative_path": (RAW_DIR / name).as_posix(), "present": present,
        "size_bytes": path.stat().st_size if present else None,
        "md5": actual_md5, "reference_md5": reference_md5,
        "source_record": RECORD_URL, "doi": RECORD_DOI,
    })

source_registry = pd.DataFrame(registry_rows)
source_registry[["file", "present", "size_bytes", "md5", "reference_md5"]]

## 5. Save the source registry and run record

Three files are written to `data/metadata/phase00/`. They are recreated on every run, so rerunning the notebook does not append duplicates.

| File | Contents |
|---|---|
| `source_registry.csv` | One row per source file: role, relative path, size, MD5 and source DOI |
| `phase00_checks.csv` | Every check from this run with its status and detail |
| `phase00_run_record.json` | Run time, Python and package versions, and the check summary |

All stored paths are relative to the project folder.

In [ ]:
out_dir = PROJECT_ROOT / OUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

check_table = pd.DataFrame(checks)
source_registry.to_csv(out_dir / "source_registry.csv", index=False)
check_table.to_csv(out_dir / "phase00_checks.csv", index=False)

run_record = {
    "phase": "00",
    "run_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "platform": platform.system(),
    "packages": {row.package: row.installed for row in packages.itertuples()},
    "source_record": RECORD_URL,
    "doi": RECORD_DOI,
    "check_counts": check_table.status.value_counts().to_dict(),
}
(out_dir / "phase00_run_record.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")

for name in ("source_registry.csv", "phase00_checks.csv", "phase00_run_record.json"):
    print(f"Saved {(OUT_DIR / name).as_posix()}")

## 6. Validation summary

The table lists every check from this run. The final line states the overall result. If any check failed, the notebook stops here with the reasons, so a failed run cannot be mistaken for a successful one.

In [ ]:
counts = check_table.status.value_counts()
n_pass, n_warn, n_fail = (int(counts.get(s, 0)) for s in ("PASS", "WARN", "FAIL"))
display(check_table)

summary = f"Phase 00: {n_pass} of {len(check_table)} checks passed, {n_warn} warning(s), {n_fail} failure(s)."
if n_fail:
    reasons = "\n".join(f"  - {r.check_id}: {r.detail}" for r in check_table[check_table.status == "FAIL"].itertuples())
    raise RuntimeError(f"{summary}\n{reasons}")
print(summary)

## 7. Findings

The first rows come from this run's checks. The remaining rows are standing scope decisions for the project; their status changes only when new evidence is added.

In [ ]:
def from_check(check_id):
    return check_table.set_index("check_id").loc[check_id, "status"]

findings = pd.DataFrame([
    {"finding": "Rheology archive and image index are identical to the files audited in Phase 01",
     "status": "Verified",  # this cell is only reached when no check failed
     "meaning": "Downstream phases read the same bytes as the published reports",
     "attention": "No"},
    {"finding": "Python packages match environment.yml",
     "status": "Verified" if from_check("P00-03") == "PASS" else "Preliminary",
     "meaning": "Numerical outputs are expected to match the reports" if from_check("P00-03") == "PASS"
                else "Workflow should run; small numerical or figure differences are possible",
     "attention": "No" if from_check("P00-03") == "PASS" else "Optional — recreate the environment from environment.yml"},
    {"finding": "images.zip available for Phase 06",
     "status": "Verified" if from_check("P00-06") == "PASS" else "Unresolved",
     "meaning": "Phase 06 image audit can run" if from_check("P00-06") == "PASS" else "Phases 00–05 can run; Phase 06 cannot",
     "attention": "No" if from_check("P00-06") == "PASS" else "Only for Phase 06 — download with --images"},
    {"finding": "Source release is Zenodo v0.0.2",
     "status": "Assumption",
     "meaning": "Version label read from the record page when the files were first downloaded; the checksums, not the label, identify the files",
     "attention": "No"},
    {"finding": "Dataset-specific reuse and redistribution terms",
     "status": "Unresolved",
     "meaning": "Raw files are not committed; derived tables and figures credit the source",
     "attention": "Yes — confirm the licence on the Zenodo record before publishing derived images"},
    {"finding": "Physical-sample identity across rheology files and images",
     "status": "Unresolved",
     "meaning": "No sample identifiers link the files; Phases 02–04 treat matches as candidates only",
     "attention": "No action in Phase 00"},
])
findings

**Handoff to Phase 01.** When every check passes, the two primary source files are confirmed present and identical to the audited versions, and their checksums are recorded in `data/metadata/phase00/source_registry.csv`. Phase 01 reads these files to audit their structure, headers, units and value ranges.